In [1]:
%cd /mnt/sdb/home/REDACTED_USER/yolov5

/mnt/sdb/home/REDACTED_USER/yolov5


In [2]:
#Direct Training 150 epoch
#It is for the stratified target dataset
#SGD and lr0=0.01
from IPython import get_ipython

cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged
export CUDA_VISIBLE_DEVICES=0,1,2,3

nohup python -m torch.distributed.run \
--nproc_per_node=4 \
train.py \
  --img 640 \
  --batch 32 \
  --epochs 150 \
  --optimizer SGD \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights yolov5n.pt \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_150 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [16]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150.log

                   all        199        904      0.779      0.676      0.733      0.387
              Birdnest        199         34      0.798      0.816      0.902      0.484
      Broken_Insulator        199         28       0.69      0.571      0.651      0.397
      Defective_Damper        199         39       0.75      0.538      0.582      0.212
   Flashover_Insulator        199         71      0.774      0.577      0.636       0.28
         Normal_Damper        199        354      0.734      0.685      0.707      0.361
     Normal_Insulators        199        289      0.806      0.817      0.815      0.458
Self-Exploded_Insulator        199         89      0.902      0.727      0.835      0.514
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150
[rank0]:[W603 02:52:46.559973225 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more 

In [17]:
#Now I will use this weight to finetune using the Oscar's parameters
#lr0=0.00334 and lrf=0.1535
from IPython import get_ipython

cmd = r"""
export CUDA_VISIBLE_DEVICES=0,1,2,3

nohup python -m torch.distributed.run \
--nproc_per_node=4 \
train.py \
  --img 640 \
  --batch 32 \
  --epochs 100 \
  --workers 8 \
  --optimizer SGD \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150/weights/best.pt \
  --hyp hyp.oscar_paper.yaml \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_150_100 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150_100.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [37]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150_100.log

                   all        199        904      0.756       0.68      0.714      0.393
              Birdnest        199         34      0.695      0.794      0.793      0.441
      Broken_Insulator        199         28      0.801        0.5      0.611      0.395
      Defective_Damper        199         39      0.682      0.564      0.589      0.259
   Flashover_Insulator        199         71      0.721       0.62      0.653      0.317
         Normal_Damper        199        354       0.72      0.711      0.716      0.372
     Normal_Insulators        199        289      0.806      0.822       0.81      0.464
Self-Exploded_Insulator        199         89       0.87      0.751      0.827      0.501
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150_100
[rank0]:[W603 03:23:39.846851333 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For m

In [38]:
#Test results...'0603_v5n_direct_merged_150_100_test'
!python val.py \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150_100/weights/best.pt \
  --img 640 \
  --batch 8 \
  --task test \
  --device cpu \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_150_100_test \
  --exist-ok

val: data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml, weights=['/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_150_100/weights/best.pt'], batch_size=8, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=cpu, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged, name=0603_v5n_direct_merged_150_100_test, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-457-g84ef1e59 Python-3.10.19 torch-2.6.0+cu118 CPU

Fusing layers... 
Model summary: 157 layers, 1768636 parameters, 0 gradients, 4.2 GFLOPs
test: Scanning /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels.cach
                 Class     Images  Instances          P          R      mAP50   
                   all        208        856      0.788      0.721      0.732   

In [6]:
#Direct Training 300 epoch
#It is for the stratified target dataset
#SGD and lr0=0.01
from IPython import get_ipython

cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged
export CUDA_VISIBLE_DEVICES=4,5,6,7

nohup python -m torch.distributed.run \
--nproc_per_node=4 \
--master_port=29501 \
train.py \
  --img 640 \
  --batch 32 \
  --epochs 300 \
  --optimizer SGD \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights yolov5n.pt \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_300 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [18]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300.log

                   all        199        904      0.793      0.697      0.737      0.414
              Birdnest        199         34      0.775      0.853       0.89       0.46
      Broken_Insulator        199         28       0.89      0.577       0.69      0.464
      Defective_Damper        199         39      0.761      0.538      0.598      0.258
   Flashover_Insulator        199         71      0.714      0.634      0.624      0.315
         Normal_Damper        199        354       0.74      0.686        0.7      0.376
     Normal_Insulators        199        289      0.797      0.851      0.841      0.493
Self-Exploded_Insulator        199         89      0.871      0.742      0.815      0.533
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300
[rank0]:[W603 03:06:11.576652318 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more 

In [23]:
#Now I will use this weight to finetune using the Oscar's parameters
#lr0=0.00334 and lrf=0.1535
from IPython import get_ipython

cmd = r"""
export CUDA_VISIBLE_DEVICES=4,5,6,7

nohup python -m torch.distributed.run \
--nproc_per_node=4 \
--master_port=29501 \
train.py \
  --img 640 \
  --batch 32 \
  --epochs 100 \
  --workers 8 \
  --optimizer SGD \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300/weights/best.pt \
  --hyp hyp.oscar_paper.yaml \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_300_100 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300_100.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [35]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300_100.log

                   all        199        904      0.782      0.698      0.735       0.41
              Birdnest        199         34      0.745      0.824      0.845       0.43
      Broken_Insulator        199         28      0.848      0.598      0.695      0.461
      Defective_Damper        199         39      0.762      0.538        0.6      0.259
   Flashover_Insulator        199         71      0.734      0.634       0.65      0.318
         Normal_Damper        199        354      0.734      0.681      0.696      0.372
     Normal_Insulators        199        289      0.805      0.848      0.844      0.495
Self-Exploded_Insulator        199         89       0.85      0.767      0.818      0.538
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300_100
[rank0]:[W603 03:24:33.223300817 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For m

In [36]:
#Test results...'0603_v5n_direct_merged_300_100_test'
!python val.py \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300_100/weights/best.pt \
  --img 640 \
  --batch 8 \
  --task test \
  --device cpu \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_300_100_test \
  --exist-ok

val: data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml, weights=['/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_300_100/weights/best.pt'], batch_size=8, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=cpu, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged, name=0603_v5n_direct_merged_300_100_test, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-457-g84ef1e59 Python-3.10.19 torch-2.6.0+cu118 CPU

Fusing layers... 
Model summary: 157 layers, 1768636 parameters, 0 gradients, 4.2 GFLOPs
test: Scanning /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels... 2
test: New cache created: /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels.cache
                 Class     Images  Instances          P         

In [60]:
#Direct Training 150 epoch Adam
#It is for the stratified target dataset
#Adam and lr0 default
from IPython import get_ipython

cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged
export CUDA_VISIBLE_DEVICES=4,5,6,7

nohup python -m torch.distributed.run \
--nproc_per_node=4 \
train.py \
  --img 640 \
  --batch 32 \
  --epochs 150 \
  --optimizer Adam \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights yolov5n.pt \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_adam_150 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [94]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150.log

                   all        199        904      0.761      0.569       0.64      0.306
              Birdnest        199         34      0.814      0.824      0.902      0.447
      Broken_Insulator        199         28      0.807      0.357      0.446      0.209
      Defective_Damper        199         39      0.687       0.41      0.479      0.135
   Flashover_Insulator        199         71      0.665      0.437      0.472      0.209
         Normal_Damper        199        354      0.749      0.585      0.661      0.311
     Normal_Insulators        199        289      0.785      0.744      0.785      0.373
Self-Exploded_Insulator        199         89      0.819      0.629      0.738      0.457
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150
[rank0]:[W603 04:12:05.423047680 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For 

In [95]:
#Now I will use this weight to finetune using the Oscar's parameters
#lr0=0.00334 and lrf=0.1535
from IPython import get_ipython

cmd = r"""
export CUDA_VISIBLE_DEVICES=0,1,2

nohup python -m torch.distributed.run \
--nproc_per_node=3 \
train.py \
  --img 640 \
  --batch 24 \
  --epochs 100 \
  --workers 8 \
  --optimizer SGD \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150/weights/best.pt \
  --hyp hyp.oscar_paper.yaml \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_adam_150_100 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150_100.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [99]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150_100.log

                   all        199        904      0.819      0.543      0.631      0.302
              Birdnest        199         34      0.909      0.824      0.911      0.453
      Broken_Insulator        199         28      0.921      0.357      0.453      0.223
      Defective_Damper        199         39      0.812      0.462       0.57      0.173
   Flashover_Insulator        199         71      0.716      0.391      0.451      0.197
         Normal_Damper        199        354      0.782      0.511      0.617      0.285
     Normal_Insulators        199        289      0.797      0.692      0.763      0.368
Self-Exploded_Insulator        199         89      0.794      0.565      0.655      0.413
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150_100
[rank0]:[W603 04:21:32.679673546 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. 

In [100]:
#Test results...'0603_v5n_direct_merged_adam_150_100_test'
!python val.py \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150_100/weights/best.pt \
  --img 640 \
  --batch 8 \
  --task test \
  --device cpu \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_adam_150_100_test \
  --exist-ok

val: data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml, weights=['/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_150_100/weights/best.pt'], batch_size=8, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=cpu, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged, name=0603_v5n_direct_merged_adam_150_100_test, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-457-g84ef1e59 Python-3.10.19 torch-2.6.0+cu118 CPU

Fusing layers... 
Model summary: 157 layers, 1768636 parameters, 0 gradients, 4.2 GFLOPs
test: Scanning /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels... 2
test: New cache created: /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels.cache
                 Class     Images  Instances          

In [74]:
#Direct Training 300 epoch Adam
#It is for the stratified target dataset
#Adam and lr0 default
from IPython import get_ipython

cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged
export CUDA_VISIBLE_DEVICES=3,4,5,6,7

nohup python -m torch.distributed.run \
--nproc_per_node=5 \
--master_port=29501 \
train.py \
  --img 640 \
  --batch 40 \
  --epochs 300 \
  --optimizer Adam \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights yolov5n.pt \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_adam_300 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [110]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300.log

                   all        199        904      0.753      0.592      0.655      0.325
              Birdnest        199         34      0.853      0.794      0.895      0.443
      Broken_Insulator        199         28      0.584      0.429      0.479      0.229
      Defective_Damper        199         39      0.723      0.564       0.54      0.207
   Flashover_Insulator        199         71      0.718      0.493       0.57      0.259
         Normal_Damper        199        354      0.773      0.568      0.645      0.315
     Normal_Insulators        199        289      0.813      0.721       0.77      0.408
Self-Exploded_Insulator        199         89       0.81      0.573      0.683      0.416
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300
[rank0]:[W603 04:31:27.989420289 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For 

In [111]:
#Now I will use this weight to finetune using the Oscar's parameters
#lr0=0.00334 and lrf=0.1535
from IPython import get_ipython

cmd = r"""
export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7

nohup python -m torch.distributed.run \
--nproc_per_node=8 \
train.py \
  --img 640 \
  --batch 64 \
  --epochs 100 \
  --workers 8 \
  --optimizer SGD \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300/weights/best.pt \
  --hyp hyp.oscar_paper.yaml \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_adam_300_100 \
  --exist-ok \
  > /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300_100.log 2>&1 &
"""
get_ipython().system_raw(cmd)

In [119]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300_100.log

                   all        199        904      0.726      0.653      0.679      0.348
              Birdnest        199         34      0.768      0.853      0.909       0.45
      Broken_Insulator        199         28      0.531      0.464      0.433      0.215
      Defective_Damper        199         39       0.69       0.59      0.548      0.198
   Flashover_Insulator        199         71      0.663      0.507      0.546      0.249
         Normal_Damper        199        354      0.772      0.642      0.684      0.349
     Normal_Insulators        199        289      0.817      0.774      0.813      0.438
Self-Exploded_Insulator        199         89      0.841      0.742      0.823      0.536
Results saved to /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300_100
[rank0]:[W603 04:37:39.893424308 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. 

In [120]:
#Test results...'0603_v5n_direct_merged_adam_300_100_test'
!python val.py \
  --data /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  --weights /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300_100/weights/best.pt \
  --img 640 \
  --batch 8 \
  --task test \
  --device cpu \
  --project /mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged \
  --name 0603_v5n_direct_merged_adam_300_100_test \
  --exist-ok

val: data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml, weights=['/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged/0603_v5n_direct_merged_adam_300_100/weights/best.pt'], batch_size=8, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=cpu, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/mnt/sdb/home/REDACTED_USER/yolov5n/runs/train/0603_v5n_direct_merged, name=0603_v5n_direct_merged_adam_300_100_test, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-457-g84ef1e59 Python-3.10.19 torch-2.6.0+cu118 CPU

Fusing layers... 
Model summary: 157 layers, 1768636 parameters, 0 gradients, 4.2 GFLOPs
test: Scanning /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels.cach
                 Class     Images  Instances          P          R      mAP50   
                   all        208        856      0.756      0.712    

In [39]:
from IPython import get_ipython
cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged
export CUDA_VISIBLE_DEVICES=0,1,2,3

nohup yolo task=detect mode=train \
  model=yolov8n.pt \
  data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  imgsz=640 \
  batch=32 \
  epochs=150 \
  optimizer=SGD \
  lr0=0.01 \
  device=0,1,2,3 \
  project=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged \
  name=0603_v8n_direct_merged_150 \
  exist_ok=True \
  > /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150.log 2>&1 &
"""

get_ipython().system_raw(cmd)

In [48]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150.log

              Birdnest         34         34      0.897      0.824      0.884      0.492
      Broken_Insulator         19         28      0.626      0.571      0.589      0.456
      Defective_Damper         30         39      0.804      0.526      0.615      0.317
   Flashover_Insulator         20         71      0.789      0.437      0.569      0.301
         Normal_Damper         90        354      0.808      0.607      0.694        0.4
     Normal_Insulators         99        289      0.824      0.794      0.836       0.52
Self-Exploded_Insulator         68         89      0.958      0.697      0.855      0.599
Speed: 0.1ms preprocess, 10.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150
💡 Learn more at https://docs.ultralytics.com/modes/train


In [49]:
from IPython import get_ipython

cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged

export CUDA_VISIBLE_DEVICES=0,1,2,3

nohup yolo task=detect mode=train \
  model=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150/weights/best.pt \
  data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  imgsz=640 \
  batch=32 \
  epochs=100 \
  workers=8 \
  optimizer=SGD \
  lr0=0.00334 \
  lrf=0.1535 \
  device=0,1,2,3 \
  project=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged \
  name=0603_v8n_direct_merged_150_100 \
  exist_ok=True \
  > /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150_100.log 2>&1 &
"""

get_ipython().system_raw(cmd)

In [57]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150_100.log

              Birdnest         34         34      0.879      0.853      0.902      0.493
      Broken_Insulator         19         28      0.638      0.536      0.583      0.466
      Defective_Damper         30         39      0.826      0.538      0.594      0.343
   Flashover_Insulator         20         71      0.687      0.535       0.58      0.312
         Normal_Damper         90        354      0.758      0.675      0.706      0.411
     Normal_Insulators         99        289      0.855      0.816      0.846       0.51
Self-Exploded_Insulator         68         89      0.919      0.762      0.868      0.574
Speed: 0.1ms preprocess, 0.3ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150_100
💡 Learn more at https://docs.ultralytics.com/modes/train


In [65]:
# Test results...'0603_v8n_direct_merged_150_100_test'
!yolo task=detect mode=val \
  model=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_150_100/weights/best.pt \
  data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  split=test \
  imgsz=640 \
  batch=8 \
  device=cpu \
  project=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged \
  name=0603_v8n_direct_merged_150_100_test \
  exist_ok=True

Ultralytics 8.4.9 🚀 Python-3.10.19 torch-2.6.0+cu118 CPU (Intel Xeon Gold 5218 CPU @ 2.30GHz)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2563.8±709.8 MB/s, size: 73.3 KB)
val: Scanning /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels... 208 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 208/208 1.1Kit/s 0.2s.4s
val: New cache created: /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 2.0it/s 13.0s0.4s
                   all        208        856      0.819      0.705       0.76       0.45
              Birdnest         34         38          1      0.912       0.98       0.54
      Broken_Insulator         20         28      0.558      0.357      0.452       0.32
      Defective_Damper         31         40      0.826      0.725      0.743      

In [40]:
from IPython import get_ipython
cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged
export CUDA_VISIBLE_DEVICES=4,5,6,7

nohup yolo task=detect mode=train \
  model=yolov8n.pt \
  data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  imgsz=640 \
  batch=32 \
  epochs=300 \
  optimizer=SGD \
  lr0=0.01 \
  device=4,5,6,7 \
  project=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged \
  name=0603_v8n_direct_merged_300 \
  exist_ok=True \
  > /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300.log 2>&1 &
"""

get_ipython().system_raw(cmd)

In [58]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300.log

              Birdnest         34         34      0.843      0.882      0.899      0.519
      Broken_Insulator         19         28      0.591      0.536      0.593      0.439
      Defective_Damper         30         39      0.689      0.512      0.549      0.262
   Flashover_Insulator         20         71      0.751      0.638      0.696      0.375
         Normal_Damper         90        354      0.767      0.697      0.724      0.407
     Normal_Insulators         99        289      0.809       0.82      0.817      0.504
Self-Exploded_Insulator         68         89      0.833      0.798      0.869      0.612
Speed: 0.1ms preprocess, 0.3ms inference, 0.0ms loss, 14.0ms postprocess per image
Results saved to /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300
💡 Learn more at https://docs.ultralytics.com/modes/train


In [59]:
from IPython import get_ipython

cmd = r"""
mkdir -p /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged

export CUDA_VISIBLE_DEVICES=0,1,2,3

nohup yolo task=detect mode=train \
  model=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300/weights/best.pt \
  data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  imgsz=640 \
  batch=32 \
  epochs=100 \
  workers=8 \
  optimizer=SGD \
  lr0=0.00334 \
  lrf=0.1535 \
  device=0,1,2,3 \
  project=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged \
  name=0603_v8n_direct_merged_300_100 \
  exist_ok=True \
  > /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300_100.log 2>&1 &
"""

get_ipython().system_raw(cmd)

In [68]:
!tail -10 /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300_100.log

              Birdnest         34         34      0.846      0.853      0.896      0.516
      Broken_Insulator         19         28      0.539      0.669      0.614      0.459
      Defective_Damper         30         39       0.68      0.513      0.573      0.271
   Flashover_Insulator         20         71      0.748      0.634      0.709      0.376
         Normal_Damper         90        354      0.742      0.726      0.736      0.408
     Normal_Insulators         99        289      0.796      0.852      0.817        0.5
Self-Exploded_Insulator         68         89      0.874      0.809       0.87      0.626
Speed: 0.1ms preprocess, 10.8ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300_100
💡 Learn more at https://docs.ultralytics.com/modes/train


In [69]:
# Test results...'0603_v8n_direct_merged_300_100_test'
!yolo task=detect mode=val \
  model=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged/0603_v8n_direct_merged_300_100/weights/best.pt \
  data=/mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/merged_stratified.yaml \
  split=test \
  imgsz=640 \
  batch=8 \
  device=cpu \
  project=/mnt/sdb/home/REDACTED_USER/yolov8n/runs/train/0603_v8n_direct_merged \
  name=0603_v8n_direct_merged_300_100_test \
  exist_ok=True

Ultralytics 8.4.9 🚀 Python-3.10.19 torch-2.6.0+cu118 CPU (Intel Xeon Gold 5218 CPU @ 2.30GHz)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2847.8±947.9 MB/s, size: 78.7 KB)
val: Scanning /mnt/sdb/home/REDACTED_USER/Merged_Dataset_Stratified/test/labels.cache... 208 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 208/208 27.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 2.2it/s 11.8s0.4s
                   all        208        856       0.76      0.725      0.749      0.459
              Birdnest         34         38      0.849      0.895      0.922      0.479
      Broken_Insulator         20         28      0.431      0.429      0.433      0.346
      Defective_Damper         31         40      0.844       0.65      0.732      0.386
   Flashover_Insulator         20         49      0.669      0.673       0.66      0.3